# MP21 — Residual CNN Super-Resolution

A residual model starts from a useful baseline and predicts a correction. A direct model predicts the entire HR image. The matched teaching comparison uses the same head, PixelShuffle tail, initialization seed, batches and update count; the skip addition is the difference.

## Mathematics, shapes and worked example

Direct: x_hat=g(y). Residual: x_hat=B(y)+g(y), with bicubic B. If target is 0.7 and B(y)=0.6, the desired residual is 0.1. A feature residual h_next=h+f(h) is distinct from this image-level skip. Its derivative includes an identity path I+df/dh. Here we isolate the image-level skip, not claim a deep residual-block experiment.

In [ ]:
from pathlib import Path
import sys,json,torch
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(ROOT/'src'))
torch.manual_seed(19)
from mp21_tools import UpsamplingCNN
x=torch.rand(1,3,5,7)
m=UpsamplingCNN(residual=True)
for p in m.parameters():p.data.zero_()
b=torch.nn.functional.interpolate(x,scale_factor=2,mode='bicubic',align_corners=False)
torch.testing.assert_close(m(x),b)
m.residual=False
assert torch.count_nonzero(m(x))==0
print('Zero residual preserves bicubic; zero direct model is black')

## Reproducible experiment

Run `python scripts/cnn_lessons_experiment.py` from the repository root. The script first checks a small synthetic input/output shape. It then uses four training and four validation NAIP scenes, split before tiling: 16 nonoverlapping 32×32 RGB patches per scene. HR stored DN are divided by 10000, not claimed as reflectance. SR inputs use Gaussian sigma 0.8 HR pixels and area x2 downsampling. Identity reconstruction uses HR as both input and target.

Training: CPU float32, two threads, seed 19, Adam learning rate 0.001, batch four, 300 updates, MSE, no augmentation or scheduler. Evaluate the final fixed step; no validation-based checkpoint search. Teaching weights are regenerated and not committed. The constant baseline uses training-channel means. SR baselines use bicubic with align_corners=False. Metrics use range one, border two, all RGB channels; equal patches per scene give equal scene weighting.

Data: existing checksummed NAIP/OpenSR crops, public domain with USDA/OpenSR attribution. No new imagery download. Final research weights/protocol/test results are untouched by these separate teaching exercises. No learner mastery claim.

## Observed result and interpretation

Matched residual learning scored 41.632 dB versus direct 34.222 dB after 300 steps. Both were below bicubic 42.964 dB. The skip helped this optimization setting but did not establish SR quality superiority. One seed does not measure training variability; the separate research study has three seeds.

In [ ]:
report=json.loads((ROOT/'reports/mp19-mp22-cnn.json').read_text())
selected=['direct_shuffle', 'residual_shuffle']
for row in report['results']:
 if row['model'] in selected:
  print({k:v for k,v in row.items() if k!='per_tile'})
from IPython.display import Image,display
display(Image(filename=str(ROOT/'reports/mp19-mp22-cnn.png')))

## Practice — solve first

1. What happens when the residual head outputs zero?
2. Does adding a skip add trainable parameters?
3. Why keep the direct model’s negative result?

## Worked solutions

1. The output is exactly the chosen interpolation baseline.
2. The addition itself adds none; surrounding layers may differ in other designs.
3. It documents the measured comparison and avoids selecting only favorable outcomes.

## Limitations and next question

One seed and a short budget describe an engineering demonstration. Interpolation, padding, input distribution and unequal parameter counts affect results. Investigate these factors on validation before any new frozen research test. Generated answers do not assess learner mastery.

[Authoritative brief](README.md) · [Local/Colab setup](../../README.md)